# Foundations of Generative AI (ICT4403) — Assignment 3
## Comparative Study of Neural Machine Translation Architectures: English → Hindi

**Language Pair:** English (Source) → Hindi (Target)  
**Dataset:** IIT Bombay English–Hindi Parallel Corpus (`cfilt/iitb-english-hindi`)  

### Architectures Evaluated:
1. **Basic Transformer** (Encoder–Decoder, trained from scratch in PyTorch)
2. **mT5-small** (Encoder–Decoder, pretrained sequence-to-sequence fine-tuned)
3. **LLaMA** (`TinyLlama-1.1B`, Decoder-only, instruction prompting)
4. **Gemma** (`Gemma-2B`, Decoder-only, in-context few-shot prompting)

---

### Step 1: Environment Setup & Library Imports

In [ ]:
# Install dependencies (if running in Colab/Fresh environment)
# !pip install -q transformers datasets sentencepiece sacrebleu accelerate peft pandas matplotlib seaborn

import os
import sys
import time
import json
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sacrebleu

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

### Step 2: Dataset Loading, Preprocessing & Exploration

In [ ]:
# Load processed datasets
train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")
test_df = pd.read_csv("../data/processed/test.csv")

print(f"Train pairs: {len(train_df):,}")
print(f"Validation pairs: {len(val_df):,}")
print(f"Test pairs: {len(test_df):,}")

# Display 10 illustrative sample pairs
test_df.head(10)

### Step 3: Model 1 — Basic Transformer (Encoder–Decoder from Scratch)

In [ ]:
sys.path.append("../models/transformer_scratch")
from tokenizer import SimpleSeq2SeqTokenizer, PAD_IDX
from model import BasicTransformer

src_tok = SimpleSeq2SeqTokenizer.load("../models/transformer_scratch/en_tokenizer.json")
tgt_tok = SimpleSeq2SeqTokenizer.load("../models/transformer_scratch/hi_tokenizer.json")

transformer_model = BasicTransformer(
    src_vocab_size=len(src_tok),
    tgt_vocab_size=len(tgt_tok),
    d_model=256,
    n_heads=4,
    num_encoder_layers=3,
    num_decoder_layers=3,
    d_ff=512,
    dropout=0.1,
    pad_idx=PAD_IDX
)
transformer_model.load_state_dict(torch.load("../models/transformer_scratch/best_model.pt", map_location="cpu"))
transformer_model.eval()

params_m1 = sum(p.numel() for p in transformer_model.parameters())
print(f"Model 1 Parameters: {params_m1:,} ({params_m1/1e6:.2f}M)")

### Step 4: Model 2 — Fine-Tuned mT5-small (Pretrained Encoder–Decoder)

In [ ]:
from transformers import AutoTokenizer as HFAutoTokenizer, AutoModelForSeq2SeqLM

mt5_dir = "../models/mt5/checkpoint"
mt5_tok = HFAutoTokenizer.from_pretrained(mt5_dir)
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(mt5_dir)
mt5_model.eval()

params_m2 = sum(p.numel() for p in mt5_model.parameters())
print(f"Model 2 (mT5-small) Parameters: {params_m2:,} ({params_m2/1e6:.2f}M)")

### Step 5: Model 3 & Model 4 — Decoder-Only LLMs (LLaMA & Gemma)

In [ ]:
sys.path.append("../models/llama")
sys.path.append("../models/gemma")
from translate_llama import LLaMATranslator
from translate_gemma import GemmaTranslator

llama_agent = LLaMATranslator()
gemma_agent = GemmaTranslator()

print(f"LLaMA Parameters: {llama_agent.param_count:,} ({llama_agent.param_count/1e9:.2f}B)")
print(f"Gemma Parameters: {gemma_agent.param_count:,} ({gemma_agent.param_count/1e9:.2f}B)")

### Step 6: Comparative Evaluation (BLEU, chrF, Latency, Parameter Counts)

In [ ]:
# Load and display model comparison results
comp_df = pd.read_csv("../results/model_comparison_metrics.csv")
comp_df

### Step 7: Visualizations & Architectural Comparison

In [ ]:
from IPython.display import Image, display

print("Figure 1: BLEU and chrF Performance Across All 4 Architectures")
display(Image("../results/plots/model_bleu_chrf_comparison.png"))

print("Figure 2: Computational Scaling: Model Size vs Inference Latency")
display(Image("../results/plots/inference_time_vs_parameters.png"))

### Step 8: Sampling-Temperature Analysis ($T = 0.0, 0.3, 0.7, 1.0$)

In [ ]:
temp_df = pd.read_csv("../results/temperature_study_results.csv")
display(temp_df)
display(Image("../results/plots/temperature_effect_curve.png"))

### Step 9: Responsible AI & Ethical Translation Examination

In [ ]:
ethical_df = pd.read_csv("../results/ethical_ai_cases.csv")
ethical_df[["Category", "English Source", "Observed Translation Behavior", "Ethical Implication / Risk", "Proposed Mitigation"]]